# Horizon Ablation — No Training

**Question:** how much predictable signal lives in correlation videos at each forecast horizon?

**Approach:** characterise the predictability of `corr_video` using baselines and statistics, varied across horizon H ∈ {1, 2, 3, 5, 10, 20, 50}. No model training — every quantity here is either a closed-form statistic or a non-learning baseline.

**What we'll learn:**
1. **Persistence MSE curve** — how fast do correlations drift over H steps?
2. **Mean-frame MSE curve** — how does a naive "just predict the average" baseline scale?
3. **Trend-extrapolation baseline** — what does simple momentum give us?
4. **Directional accuracy ceiling from trend-following** — upper bound on dir_acc without learning
5. **Autocorrelation of frame deltas** — information half-life of the change signal
6. **Per-pair predictability distribution** — are a few pairs doing all the work?

**Why this matters:** our learned models plateau at val_mse ≈ 0.069, dir_acc ≈ 73% across many architectures. We need to know whether that's the *model's* fault or the *data's* fault. This ablation answers that.

In [ ]:
# Cell 1 – Imports & Load Data
import os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

_cwd        = os.getcwd()
PROJECT_DIR = os.path.abspath(os.path.join(_cwd, '..')) \
              if os.path.basename(_cwd) == 'notebooks' else _cwd
DATA_DIR    = os.path.join(PROJECT_DIR, 'data', 'processed')
PLOTS_DIR   = os.path.join(PROJECT_DIR, 'outputs', 'plots')

corr_video = np.load(os.path.join(DATA_DIR, 'corr_video.npy'))
with open(os.path.join(DATA_DIR, 'pair_meta.pkl'), 'rb') as f:
    pair_meta = pickle.load(f)
tickers = pair_meta['tickers']

T, N, _ = corr_video.shape
print(f'corr_video: {corr_video.shape}  (T={T} snapshots, N={N} stocks)')

# Same train/val split as the trials so results are comparable
LOOKBACK    = 20
TRAIN_RATIO = 0.70

HORIZONS = [1, 2, 3, 5, 10, 20, 50]
print(f'Horizons to test: {HORIZONS}')


In [ ]:
# Cell 2 – Reproduce the Same Train/Val Split Used in Trials 2-6
# So our baselines are evaluated on the SAME val snapshots the models saw.

# Use H=5 (the trial horizon) for the split construction, then re-evaluate on all H.
H_SPLIT = 5
n_valid = T - max(HORIZONS)        # ensure room for the longest horizon
frame_mse = np.array([
    float(np.mean((corr_video[t] - corr_video[t + H_SPLIT]) ** 2))
    for t in range(n_valid)
], dtype=np.float32)

volatility_threshold = float(np.median(frame_mse))
valid_snaps  = np.array([s for s in range(n_valid) if s >= LOOKBACK])
snap_v       = frame_mse[valid_snaps]
volatile_snaps = valid_snaps[snap_v >= volatility_threshold]
stable_snaps   = valid_snaps[snap_v <  volatility_threshold]

np.random.seed(42)
vol_train = np.random.choice(volatile_snaps, int(len(volatile_snaps) * TRAIN_RATIO), replace=False)
vol_val   = np.setdiff1d(volatile_snaps, vol_train)
sta_train = np.random.choice(stable_snaps,  int(len(stable_snaps)  * TRAIN_RATIO), replace=False)
sta_val   = np.setdiff1d(stable_snaps, sta_train)

train_snaps = np.sort(np.concatenate([vol_train, sta_train]))
val_snaps   = np.sort(np.concatenate([vol_val,   sta_val]))

mean_train_frame = corr_video[train_snaps].mean(axis=0)
print(f'Train: {len(train_snaps)}  Val: {len(val_snaps)}')
print(f'(Same stratified split used in Trials 2-6, so baselines are directly comparable.)')


In [ ]:
# Cell 3 – Baseline MSE Curves Across Horizons
# Three baselines per horizon:
#   1. Persistence:   predict corr[s+H] = corr[s]   (no change assumed)
#   2. Mean-frame:    predict corr[s+H] = mean of training frames
#   3. Trend extrap:  predict corr[s+H] = corr[s] + H * (corr[s] - corr[s-K]) / K
#                     using a short window K (captures local momentum)

K_TREND = 3   # use 3-step trend

def compute_baselines(H, K=K_TREND):
    # Only use val snapshots where we have both the lookback for trend AND the future for H
    sn = np.array([s for s in val_snaps if s + H < T and s - K >= 0])

    # 1. Persistence
    persist_mse = np.mean([
        np.mean((corr_video[s] - corr_video[s + H]) ** 2) for s in sn
    ])

    # 2. Mean-frame
    mean_mse = np.mean([
        np.mean((mean_train_frame - corr_video[s + H]) ** 2) for s in sn
    ])

    # 3. Trend extrapolation
    trend_mse = np.mean([
        np.mean((corr_video[s] + H * (corr_video[s] - corr_video[s - K]) / K
                 - corr_video[s + H]) ** 2)
        for s in sn
    ])

    return dict(n=len(sn), persistence=persist_mse, mean_frame=mean_mse,
                trend=trend_mse)

rows = []
for H in HORIZONS:
    b = compute_baselines(H)
    rows.append(dict(H=H, n=b['n'],
                     persistence=b['persistence'],
                     mean_frame=b['mean_frame'],
                     trend=b['trend'],
                     best_of_three=min(b['persistence'], b['mean_frame'], b['trend'])))

baseline_df = pd.DataFrame(rows)
print(baseline_df.to_string(index=False, float_format='%.6f'))


In [ ]:
# Cell 4 - Directional Accuracy Baselines (with mean-reversion variants)
# We test BOTH directions of the trend baseline. If correlations exhibit mean
# reversion, the *inverted* trend will outperform the direct trend.

def dir_acc_baselines(H, K=K_TREND, mag_thresh=0.005):
    sn = np.array([s for s in val_snaps if s + H < T and s - K >= 0])
    last_correct, trend_correct = 0, 0
    last_correct_masked, trend_correct_masked = 0, 0
    masked_n, total_n = 0, 0
    for s in sn:
        true_delta = corr_video[s + H] - corr_video[s]
        last_delta = corr_video[s]     - corr_video[s - 1]
        trend_delt = corr_video[s]     - corr_video[s - K]
        last_correct  += np.sum(np.sign(last_delta) == np.sign(true_delta))
        trend_correct += np.sum(np.sign(trend_delt) == np.sign(true_delta))
        total_n       += true_delta.size
        mask = (np.abs(true_delta) > mag_thresh)
        masked_n += mask.sum()
        last_correct_masked  += np.sum((np.sign(last_delta) == np.sign(true_delta)) & mask)
        trend_correct_masked += np.sum((np.sign(trend_delt) == np.sign(true_delta)) & mask)
    last_dir   = last_correct  / max(1, total_n)
    trend_dir  = trend_correct / max(1, total_n)
    last_dir_m = last_correct_masked  / max(1, masked_n)
    trend_dir_m= trend_correct_masked / max(1, masked_n)
    return dict(
        n_samples=len(sn),
        # direct (momentum) baseline
        last_dir=last_dir, trend_dir=trend_dir,
        last_dir_masked=last_dir_m, trend_dir_masked=trend_dir_m,
        # inverted (mean-reversion) baseline = 1 - direct
        inv_last_dir=1 - last_dir, inv_trend_dir=1 - trend_dir,
        inv_last_dir_masked=1 - last_dir_m, inv_trend_dir_masked=1 - trend_dir_m,
    )

dir_rows = []
for H in HORIZONS:
    d = dir_acc_baselines(H)
    dir_rows.append(dict(H=H,
                         momentum_last=d["last_dir_masked"],
                         momentum_trend=d["trend_dir_masked"],
                         meanrev_last=d["inv_last_dir_masked"],
                         meanrev_trend=d["inv_trend_dir_masked"]))
dir_df = pd.DataFrame(dir_rows)
print(dir_df.to_string(index=False, float_format="%.4f"))
print()
print("momentum_*  : predict direction = sign of past delta (momentum)")
print("meanrev_*   : predict direction = OPPOSITE of past delta (mean reversion)")
print("All values are magnitude-masked (|true_delta| > 0.005)")
print("-> If meanrev > 0.5 and momentum < 0.5, correlations mean-revert at that H.")


In [ ]:
# Cell 5 – Autocorrelation of Frame Deltas (Information Half-Life)
# For each pair (i,j), the time series of single-step deltas delta[t] = corr[t]-corr[t-1].
# autocorr(delta, H) tells us: do today's changes predict changes H steps from now?
#   > 0 : trend persists (momentum)
#   = 0 : random walk (no predictability beyond mean)
#   < 0 : mean reversion (overshoot)

delta_video = np.diff(corr_video, axis=0)       # (T-1, N, N)
mean_delta  = delta_video.mean(axis=0)
delta_demean = delta_video - mean_delta

# Compute per-pair autocorrelation at each lag, then average across pairs
# (averaging keeps signal even when individual pairs are noisy)
var_per_pair = (delta_demean ** 2).mean(axis=0) + 1e-10   # (N, N)

autocorr_results = []
for H in HORIZONS:
    cov = (delta_demean[:-H] * delta_demean[H:]).mean(axis=0)   # (N, N)
    autocorr = cov / var_per_pair
    autocorr_results.append(dict(
        H=H,
        mean_autocorr=float(autocorr.mean()),
        median_autocorr=float(np.median(autocorr)),
        frac_positive=float((autocorr > 0).mean()),
        frac_strong=float((np.abs(autocorr) > 0.05).mean()),
    ))
ac_df = pd.DataFrame(autocorr_results)
print(ac_df.to_string(index=False, float_format='%.4f'))
print('\nReading guide:')
print('  mean_autocorr ~ 0   -> information is gone, no predictability from past changes')
print('  mean_autocorr > 0   -> momentum present (trends persist)')
print('  mean_autocorr < 0   -> mean reversion')


In [ ]:
# Cell 6 – Per-Pair Predictability Distribution
# Are a few pairs doing all the lifting? Histogram how predictable each pair is.

# Predictability proxy per pair: variance of its correlation series.
# A pair whose correlation barely moves is "easy" - persistence already nails it.
# A pair whose correlation swings wildly is "hard".

pair_var = corr_video.var(axis=0)                  # (N, N)
pair_var_flat = pair_var[np.triu_indices(N, k=1)]  # unique pairs only

# Persistence error per pair at H=5
H_REF = 5
persist_err = np.array([
    ((corr_video[s] - corr_video[s + H_REF]) ** 2) for s in val_snaps if s + H_REF < T
]).mean(axis=0)                                    # (N, N) MSE per pair
persist_err_flat = persist_err[np.triu_indices(N, k=1)]

# Mean-frame error per pair at H=5
mean_err = np.array([
    ((mean_train_frame - corr_video[s + H_REF]) ** 2) for s in val_snaps if s + H_REF < T
]).mean(axis=0)
mean_err_flat = mean_err[np.triu_indices(N, k=1)]

print(f'Per-pair MSE at H={H_REF}  (N_pairs={len(pair_var_flat)})')
print(f'  Persistence: median={np.median(persist_err_flat):.5f}  '
      f'95th pct={np.percentile(persist_err_flat, 95):.5f}')
print(f'  Mean-frame : median={np.median(mean_err_flat):.5f}    '
      f'95th pct={np.percentile(mean_err_flat, 95):.5f}')

# How concentrated is the error? Top-5% of pairs contribute what fraction of total MSE?
n_top = int(0.05 * len(persist_err_flat))
top5_persist = np.sort(persist_err_flat)[-n_top:].sum() / persist_err_flat.sum()
top5_mean    = np.sort(mean_err_flat   )[-n_top:].sum() / mean_err_flat   .sum()
print(f'\nConcentration: top 5% of pairs account for ...')
print(f'  {top5_persist*100:.1f}% of persistence MSE')
print(f'  {top5_mean*100:.1f}% of mean-frame MSE')
print('\n(If a small fraction of pairs dominates the error, focused work on those pairs would help.)')


In [ ]:
# Cell 7 - Reference Trial 5 Performance Against Baselines

T5_MSE   = 0.068600
T5_DIR   = 0.730
T5_PEAR  = 0.3724

H = 5
b5  = compute_baselines(H)
d5  = dir_acc_baselines(H)

print(f"COMPARISON AT H={H} (where Trial 5 model was trained)")
print("=" * 72)
header = f"{'Method':<35} {'MSE':>10} {'Dir Acc':>10} {'vs Persist':>12}"
print(header)
print("-" * 72)

def show(name, mse, dir_acc):
    base = b5["persistence"]
    if mse is not None:
        vs = (base - mse) / base * 100
        mse_s = f"{mse:.6f}"
        vs_s  = f"{vs:+.1f}%"
    else:
        mse_s, vs_s = "   -    ", "   -  "
    dir_s = f"{dir_acc:.4f}" if dir_acc is not None else "  -   "
    print(f"{name:<35} {mse_s:>10} {dir_s:>10} {vs_s:>12}")

show("Persistence",                  b5["persistence"], 0.5)
show("Mean-frame",                   b5["mean_frame"],  None)
show("Trend extrap (K=3) MSE",       b5["trend"],       None)
show("Momentum (last-delta) dir",    None, d5["last_dir_masked"])
show("Momentum (3-step) dir",        None, d5["trend_dir_masked"])
show("Mean-reversion (last-delta)",  None, d5["inv_last_dir_masked"])
show("Mean-reversion (3-step)",      None, d5["inv_trend_dir_masked"])
print("-" * 72)
show("Trial 5 ConvLSTM (learned)",   T5_MSE, T5_DIR)

mr_dir = d5["inv_last_dir_masked"]
gap_mse = b5["mean_frame"] - T5_MSE
gap_dir = T5_DIR - mr_dir
print()
print("HEADROOM ANALYSIS:")
print(f"  Mean-frame MSE - Trial 5 MSE     : {gap_mse:+.6f}  ({gap_mse/b5['mean_frame']*100:+.1f}% of mean-frame MSE)")
print(f"  Mean-reversion dir_acc (ceiling) : {mr_dir:.4f}")
print(f"  Trial 5 dir_acc                  : {T5_DIR:.4f}")
print(f"  Gain over mean-reversion baseline: {gap_dir*100:+.1f}pp")


In [ ]:
# Cell 8 - Plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Horizon Ablation - Predictability of Correlation Videos",
             fontsize=14, fontweight="bold")

# (1) MSE baselines vs H (skip trend extrap, it explodes)
ax = axes[0, 0]
ax.plot(baseline_df["H"], baseline_df["persistence"], "o-", label="Persistence (predict no change)",
        color="steelblue")
ax.plot(baseline_df["H"], baseline_df["mean_frame"], "s-", label="Mean-frame (predict avg)",
        color="darkorange")
ax.axhline(T5_MSE, color="crimson", linestyle="--",
           label="Trial 5 ConvLSTM (H=5)")
ax.set_xlabel("Horizon H"); ax.set_ylabel("Val MSE")
ax.set_title("MSE baselines vs horizon")
ax.set_xscale("log"); ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# (2) Directional baselines vs H
ax = axes[0, 1]
ax.plot(dir_df["H"], dir_df["momentum_last"],  "o-",  label="Momentum (1-step)",      color="steelblue")
ax.plot(dir_df["H"], dir_df["momentum_trend"], "s-",  label="Momentum (3-step)",      color="darkorange")
ax.plot(dir_df["H"], dir_df["meanrev_last"],   "o--", label="Mean-reversion (1-step)", color="steelblue", alpha=0.6)
ax.plot(dir_df["H"], dir_df["meanrev_trend"],  "s--", label="Mean-reversion (3-step)", color="darkorange", alpha=0.6)
ax.axhline(T5_DIR, color="crimson", linestyle="--", label="Trial 5 dir_acc (H=5)")
ax.axhline(0.5,    color="grey",    linestyle=":",  label="Random (0.5)")
ax.set_xlabel("Horizon H"); ax.set_ylabel("Directional Accuracy")
ax.set_title("Directional-accuracy baselines vs horizon")
ax.set_xscale("log"); ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# (3) Autocorrelation decay
ax = axes[1, 0]
ax.plot(ac_df["H"], ac_df["mean_autocorr"],   "o-", label="Mean autocorr",   color="steelblue")
ax.plot(ac_df["H"], ac_df["median_autocorr"], "s-", label="Median autocorr", color="darkorange")
ax.axhline(0.0, color="grey", linestyle=":")
ax.set_xlabel("Horizon H"); ax.set_ylabel("Autocorrelation of frame-deltas")
ax.set_title("Information half-life of correlation changes")
ax.set_xscale("log"); ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# (4) Per-pair MSE distribution
ax = axes[1, 1]
ax.hist(persist_err_flat, bins=80, alpha=0.5, label="Persistence per-pair MSE @ H=5", color="steelblue")
ax.hist(mean_err_flat,    bins=80, alpha=0.5, label="Mean-frame per-pair MSE @ H=5", color="darkorange")
ax.set_yscale("log")
ax.set_xlabel("Per-pair MSE"); ax.set_ylabel("Number of pairs")
ax.set_title("Predictability is highly heterogeneous across pairs")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "horizon_ablation.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved horizon_ablation.png")


In [ ]:
# Cell 9 - Summary & Interpretation

crossover = None
for i, row in baseline_df.iterrows():
    if row["persistence"] > row["mean_frame"]:
        crossover = row["H"]
        break

print("SUMMARY")
print("=" * 70)
print()
print(f"Persistence/Mean-frame crossover horizon: H ~ {crossover}")
print(f"  -> beyond H={crossover}, persistence is worse than mean-frame")
print()
print("Information half-life (mean autocorr |.|< 0.05):")
ac_lt = ac_df[ac_df["mean_autocorr"].abs() < 0.05]
if len(ac_lt) > 0:
    print(f"  -> reached at H ~ {ac_lt.iloc[0]['H']}  (autocorr = {ac_lt.iloc[0]['mean_autocorr']:.4f})")
else:
    print(f"  -> not reached within tested horizons; signal persists longer than H={HORIZONS[-1]}")
print()
print("At Trial 5 horizon (H=5):")
b5_row = baseline_df[baseline_df["H"] == 5].iloc[0]
d5_row = dir_df[dir_df["H"] == 5].iloc[0]
gap_to_T5 = b5_row["mean_frame"] - T5_MSE
print(f"  MSE: mean_frame={b5_row['mean_frame']:.6f}, Trial 5={T5_MSE:.6f}, gap={gap_to_T5:.6f}")
print(f"  Dir: random=0.500, mean-reversion={d5_row['meanrev_last']:.4f}, Trial 5={T5_DIR:.4f}")
print(f"  Model adds {(T5_DIR - d5_row['meanrev_last'])*100:+.1f}pp over the mean-reversion baseline")
print()
print("KEY FINDINGS:")
print("-" * 70)
print("1. Mean-frame MSE is FLAT across H (0.071-0.073) - the long-term average is a")
print("   surprisingly hard baseline to beat at any horizon.")
print("2. Persistence MSE is HIGH (0.13+) at all H - even consecutive snapshots differ")
print("   substantially. Correlations are noisy frame-to-frame.")
print("3. Frame-delta autocorr at H=1 = -0.47 - STRONG single-step mean reversion.")
print("   Beyond H=2, autocorr ~ 0 - signal decorrelates almost immediately.")
print("4. Mean-reversion baseline gets 66% dir_acc at ALL horizons (data is symmetric).")
print("5. Learned model gets 73% dir_acc - adds ~7pp over the mean-reversion ceiling.")
print()
print("IMPLICATIONS:")
print("-" * 70)
print("* The 73% wall is essentially: 66% mean-reversion + 7% learned regime/cluster")
print("  patterns. With this input, 80%+ is unlikely without richer signals.")
print("* Most predictive signal lives at H=1. Beyond H=2 the changes are noise.")
print("* The model is essentially a sophisticated mean-reversion engine with a few")
print("  extra percentage points from learned cluster/regime structure.")
